# Spatial Distances

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import os
import tifffile
import geopandas as gpd
from skimage import filters

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Funs

In [ ]:
def img_map(img,figsize=(3,3),cmap='viridis',title=None, origin='lower',labels=False,cbar=False,save=None):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(img, cmap=cmap, origin=origin)
    ax.set_title(title)
    if not labels:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    if cbar:
        plt.colorbar(im, ax=ax)
    if save is not None:
        plt.savefig(save, dpi=300, bbox_inches='tight', transparent=False) 
    plt.show()

def pctnorm(img, pct_lo = 1, pct_hi = 99.7):
    from skimage import exposure
    p_low, p_high = np.percentile(img, [pct_lo, pct_hi])
    return exposure.rescale_intensity(img, in_range=(p_low, p_high), out_range=(0, 1))

def mask2shapely(masks_in):
    from shapely.geometry import Polygon
    import geopandas as gpd
    from skimage import measure
    import numpy as np
    
    ids = []
    geometries = []

    for i in np.unique(masks_in)[1:]:
        binary_mask = (masks_in == i).astype(np.uint8)
        
        # Find contours using scikit-image
        contours = measure.find_contours(binary_mask, 0.5)
        
        # If multiple contours (disconnected pixels), take the longest one
        if len(contours) > 1:
            longest_contour = max(contours, key=len)
            # Convert from (row, col) to (x, y) coordinates
            coords = longest_contour[:, [1, 0]]
            if len(coords) > 3:
                geometries.append(Polygon(coords))
                ids.append(i)
        else:
            if len(contours) > 0 and len(contours[0]) > 3:
                # Convert from (row, col) to (x, y) coordinates
                coords = contours[0][:, [1, 0]]
                geometries.append(Polygon(coords))
                ids.append(i)
    
    gdf = gpd.GeoDataFrame({'id': ids, 'geometry': geometries}, crs='EPSG:4326')
    gdf = gdf[gdf['id'] != 0]
    gdf['x'] = gdf.geometry.centroid.x
    gdf['y'] = gdf.geometry.centroid.y   

    return gdf

def mask_map(img,mask, figsize=(4,4),fs=None, linewidth=0.75, title=None, origin='upper', cmap='viridis', labels=False):
    if fs is not None:
        figsize=fs


    gdf = mask2shapely(mask)
    
    fig,ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap=cmap, interpolation='none', origin=origin)
    gdf.boundary.plot(aspect=1, ax=ax, color='white',linewidth=linewidth)
    ax.set_title(title)
    if not labels:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from scipy.spatial.distance import cdist

def calculate_average_border_distances_centroids(df, max_distance=141):
    """
    Ultra-fast centroid-based distance calculation.
    
    Args:
        df: DataFrame with columns ['cell_id', 'x', 'y'] (or 'X', 'Y')
        max_distance: maximum distance threshold
    
    Returns:
        border_distance_matrix: scipy sparse matrix
        cell_ids: numpy array of cell IDs
    """
    # Get coordinates (handle different column names)
    if 'x' in df.columns:
        coords = df[['Cell_ID', 'x', 'y']].values
    else:
        coords = df[['Cell_ID', 'X', 'Y']].values
    
    # Get unique cells and their centroids
    cell_ids = np.unique(coords[:, 0])
    cell_ids = np.sort(cell_ids[cell_ids != 0])
    n_cells = len(cell_ids)
    
    centroids = np.zeros((n_cells, 2))
    for i, cell_id in enumerate(cell_ids):
        mask = coords[:, 0] == cell_id
        centroids[i] = coords[mask, 1:].mean(axis=0)
    
    # Compute all pairwise distances at once
    distances = cdist(centroids, centroids, metric='euclidean')
    
    # Create sparse matrix: use actual distance if <= max_distance, else -1
    mask = (distances > 0) & (distances <= max_distance)
    distances_filtered = np.where(mask, distances, -1)
    
    # Convert to COO format (only store non-background pairs)
    rows, cols = np.where(distances_filtered != -1)
    data = distances_filtered[rows, cols]
    
    border_matrix = coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells))
    
    return border_matrix, cell_ids

def coo_to_long_df_vectorized(border_matrix, cell_ids, df, cell_id='Cell_ID', phenotype_col='ph', additional_cols=None):
    """
    Ultra-fast vectorized version using pandas merge with support for multiple metadata columns.
    
    Args:
        border_matrix: scipy.sparse.coo_matrix from distance calculation
        cell_ids: array of cell IDs corresponding to matrix indices
        df: DataFrame with cell_id and metadata columns
        phenotype_col: name of phenotype column in df
        additional_cols: list of additional column names to attach, or None
    
    Returns:
        DataFrame with columns: cell_id_1, cell_id_2, distance, phenotype_1, phenotype_2, 
                                and any additional columns as col_name_1, col_name_2
    """
    # Get COO matrix data
    border_matrix_coo = border_matrix.tocoo()
    
    # Create base dataframe
    result_df = pd.DataFrame({
        'cell_id_1': cell_ids[border_matrix_coo.row],
        'cell_id_2': cell_ids[border_matrix_coo.col],
        'distance': border_matrix_coo.data
    })
    
    # Remove duplicate pairs (keep only upper triangle)
    ##result_df = result_df[result_df['cell_id_1'] < result_df['cell_id_2']].copy()
    
    # Determine columns to merge
    cols_to_merge = [cell_id, phenotype_col]
    if additional_cols is not None:
        if isinstance(additional_cols, str):
            additional_cols = [additional_cols]
        cols_to_merge.extend(additional_cols)
    
    # Get lookup table with all relevant columns
    lookup = df[cols_to_merge].drop_duplicates()
    
    # Merge for cell_id_1
    result_df = result_df.merge(
        lookup, 
        left_on='cell_id_1', 
        right_on=cell_id, 
        how='left'
    ).drop(columns=[cell_id])
    
    # Rename columns with _1 suffix
    rename_dict = {phenotype_col: 'phenotype_1'}
    if additional_cols is not None:
        for col in additional_cols:
            rename_dict[col] = f'{col}_1'
    result_df = result_df.rename(columns=rename_dict)
    
    # Merge for cell_id_2
    result_df = result_df.merge(
        lookup, 
        left_on='cell_id_2', 
        right_on=cell_id, 
        how='left'
    ).drop(columns=[cell_id])
    
    # Rename columns with _2 suffix
    rename_dict = {phenotype_col: 'phenotype_2'}
    if additional_cols is not None:
        for col in additional_cols:
            rename_dict[col] = f'{col}_2'
    result_df = result_df.rename(columns=rename_dict)
    
    return result_df


In [ ]:
#1414px=2000um
#1414/10px = 141px = 2000/10 = 200um
#1414/10px = 200 px = 2000/10 = 200um

In [ ]:
1414/200

### PostTP

In [ ]:
sorted_pairs
out_dict = {}

cn_id = "postTP_linNegOut_15clus"
cn_cells_data = pd.read_csv(f'/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/{cn_id}-cells_data.csv')
obs_data = pd.read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/obs_v5.csv')

from tqdm import tqdm

for sample_id in tqdm(cn_cells_data['sample_id'].unique()):

    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]    
    #GATE CELLS
    obs_sample = obs_sample[obs_sample.gated]

    #merge cn
    cn_cells_sample = cn_cells_data[cn_cells_data.sample_id==sample_id]
    obs_sample = obs_sample.merge(cn_cells_sample[['Cell_ID','cn_labels']],how='left')

    # Standard version (very fast for <10k cells)
    border_matrix, cell_ids = calculate_average_border_distances_centroids(obs_sample, max_distance=141)

    long_df = coo_to_long_df_vectorized(
        border_matrix, cell_ids, obs_sample, 
        phenotype_col='ph',
        additional_cols='cn_labels'
    )
    #only compare WITHIN CN
    long_df = long_df[long_df.cn_labels_1==long_df.cn_labels_2]
    
    long_df = (long_df
        .loc[long_df.groupby(['cell_id_1', 'phenotype_2'])['distance'].idxmin()]
        .pivot(index=['cell_id_1', 'phenotype_1', 'cn_labels_1'], 
               columns='phenotype_2', 
               values='distance')
        .reset_index()
    )

    long_df['sample_id'] = sample_id
    
    out_dict[sample_id] = long_df

long_df = pd.concat(out_dict.values(),axis=0)
long_df = long_df.rename_axis(None, axis=1).reset_index().drop('index',axis=1)
long_df = long_df[['sample_id','cell_id_1', 'phenotype_1', 'cn_labels_1']+sorted(cn_cells_data['ph'].unique().tolist())]
long_df = long_df.rename({'cell_id_1':'index','phenotype_1':'labels','cn_labels_1':'cn_celltypes'},axis=1)

saveloc = "/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/"
long_df.to_csv(f"{saveloc}{cn_id}-distances.csv",index=False)

### PreTP

In [ ]:

out_dict = {}

cn_id = "bslTP_linNegOut_15clus"
cn_cells_data = pd.read_csv(f'/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/{cn_id}-cells_data.csv')
obs_data = pd.read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/obs_v5.csv')

from tqdm import tqdm

for sample_id in tqdm(cn_cells_data['sample_id'].unique()):

    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]    
    #GATE CELLS
    obs_sample = obs_sample[obs_sample.gated]

    #merge cn
    cn_cells_sample = cn_cells_data[cn_cells_data.sample_id==sample_id]
    obs_sample = obs_sample.merge(cn_cells_sample[['Cell_ID','cn_labels']],how='left')

    # Standard version (very fast for <10k cells)
    border_matrix, cell_ids = calculate_average_border_distances_centroids(obs_sample, max_distance=141)

    long_df = coo_to_long_df_vectorized(
        border_matrix, cell_ids, obs_sample, 
        phenotype_col='ph',
        additional_cols='cn_labels'
    )
    #only compare WITHIN CN
    long_df = long_df[long_df.cn_labels_1==long_df.cn_labels_2]
    
    long_df = (long_df
        .loc[long_df.groupby(['cell_id_1', 'phenotype_2'])['distance'].idxmin()]
        .pivot(index=['cell_id_1', 'phenotype_1', 'cn_labels_1'], 
               columns='phenotype_2', 
               values='distance')
        .reset_index()
    )

    long_df['sample_id'] = sample_id
    
    out_dict[sample_id] = long_df

long_df = pd.concat(out_dict.values(),axis=0)
long_df = long_df.rename_axis(None, axis=1).reset_index().drop('index',axis=1)
long_df = long_df[['sample_id','cell_id_1', 'phenotype_1', 'cn_labels_1']+sorted(cn_cells_data['ph'].unique().tolist())]
long_df = long_df.rename({'cell_id_1':'index','phenotype_1':'labels','cn_labels_1':'cn_celltypes'},axis=1)

saveloc = "/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/"
long_df.to_csv(f"{saveloc}{cn_id}-distances.csv",index=False)

### Inter-CN contact

#### Pre-TP

In [ ]:
#1414px=2000um
#1414/200px = 7 = 2000/200 = 10um

In [ ]:

out_dict = {}

cn_id = "bslTP_linNegOut_15clus"

cn_cells_data = pd.read_csv(f'/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/{cn_id}-cells_data.csv')
obs_data = pd.read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/obs_v5.csv')

from tqdm import tqdm

for sample_id in tqdm(cn_cells_data['sample_id'].unique()):

    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]    
    #GATE CELLS
    obs_sample = obs_sample[obs_sample.gated]

    #merge cn
    cn_cells_sample = cn_cells_data[cn_cells_data.sample_id==sample_id]
    obs_sample = obs_sample.merge(cn_cells_sample[['Cell_ID','cn_labels']],how='left')

    # Standard version (very fast for <10k cells)
    border_matrix, cell_ids = calculate_average_border_distances_centroids(obs_sample, max_distance=7)

    long_df = coo_to_long_df_vectorized(
        border_matrix, cell_ids, obs_sample, 
        phenotype_col='cn_labels',
        additional_cols=None
    )
    
    long_df['sample_id'] = sample_id
    
    out_dict[sample_id] = long_df

    
long_df = pd.concat(out_dict.values(),axis=0)
saveloc="/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/"
long_df.to_csv(f"{saveloc}{cn_id}-cn_dist.csv",index=False)

In [ ]:
long_df

In [ ]:
# Raw overlap

long_df_ = long_df[(~long_df.phenotype_1.isna())&(~long_df.phenotype_2.isna())]
sorted_pairs = pd.DataFrame(
    np.sort(long_df_[['phenotype_1', 'phenotype_2']].values, axis=1),
    columns=['ph_min', 'ph_max']
)
sorted_pairs=sorted_pairs.value_counts().reset_index()

data = sorted_pairs.pivot(index='ph_min', columns='ph_max', values='count')

sns.clustermap(data.fillna(data.T), 
               z_score=None, figsize=(6,6), xticklabels=True, yticklabels=True, col_cluster=True, row_cluster=True, 
               dendrogram_ratio=0.2, linewidths=0.5, cmap='Reds', linecolor='white')
plt.savefig(f"{saveloc}{cn_id}-hm_N.pdf", dpi=300, bbox_inches='tight') # Saves with high resolution and minimal whitespace
plt.show() 

In [ ]:
# Log overlap

long_df_ = long_df[(~long_df.phenotype_1.isna())&(~long_df.phenotype_2.isna())]
sorted_pairs = pd.DataFrame(
    np.sort(long_df_[['phenotype_1', 'phenotype_2']].values, axis=1),
    columns=['ph_min', 'ph_max']
)
sorted_pairs=sorted_pairs.value_counts().reset_index()

data = sorted_pairs.pivot(index='ph_min', columns='ph_max', values='count')

sns.clustermap(np.log10(data.fillna(data.T)), 
               z_score=None, figsize=(6,6), xticklabels=True, yticklabels=True, col_cluster=True, row_cluster=True, 
               dendrogram_ratio=0.1, linewidths=0.5, cmap='Reds', linecolor='white')
plt.savefig(f"{saveloc}{cn_id}-hm_log10N.pdf", dpi=300, bbox_inches='tight') # Saves with high resolution and minimal whitespace
plt.show()

In [ ]:
# Normalize % all cells

phenotype_totals = pd.concat([
    sorted_pairs.groupby(['ph_min'])['count'].sum(),
    sorted_pairs.groupby(['ph_max'])['count'].sum()
]).groupby(level=0).sum()

sorted_pairs['normalized'] = sorted_pairs.apply(
    lambda row: row['count'] / np.sqrt(
        phenotype_totals[row['ph_min']] * phenotype_totals[row['ph_max']]
    ) * 100, axis=1
)
data = sorted_pairs.pivot(index='ph_min', columns='ph_max', values='normalized')


sns.clustermap(data.fillna(data.T), 
               z_score=None, figsize=(6,6), xticklabels=True, yticklabels=True, col_cluster=True, row_cluster=True, 
               dendrogram_ratio=0.2, linewidths=0.5, cmap='Reds', linecolor='white')
plt.savefig(f"{saveloc}{cn_id}-hm_pct.pdf", dpi=300, bbox_inches='tight') # Saves with high resolution and minimal whitespace


#### PostTP

In [ ]:
#1414px=2000um
#1414/200px = 7 = 2000/200 = 10um

In [ ]:

out_dict = {}

cn_id = "postTP_linNegOut_15clus"

cn_cells_data = pd.read_csv(f'/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/{cn_id}-cells_data.csv')
obs_data = pd.read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/obs_v5.csv')

from tqdm import tqdm

for sample_id in tqdm(cn_cells_data['sample_id'].unique()):

    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]    
    #GATE CELLS
    obs_sample = obs_sample[obs_sample.gated]

    #merge cn
    cn_cells_sample = cn_cells_data[cn_cells_data.sample_id==sample_id]
    obs_sample = obs_sample.merge(cn_cells_sample[['Cell_ID','cn_labels']],how='left')

    # Standard version (very fast for <10k cells)
    border_matrix, cell_ids = calculate_average_border_distances_centroids(obs_sample, max_distance=7)

    long_df = coo_to_long_df_vectorized(
        border_matrix, cell_ids, obs_sample, 
        phenotype_col='cn_labels',
        additional_cols=None
    )
    
    long_df['sample_id'] = sample_id
    
    out_dict[sample_id] = long_df

    
long_df = pd.concat(out_dict.values(),axis=0)
saveloc="/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/"
long_df.to_csv(f"{saveloc}{cn_id}-cn_dist.csv",index=False)

In [ ]:
# Raw overlap

long_df_ = long_df[(~long_df.phenotype_1.isna())&(~long_df.phenotype_2.isna())]
sorted_pairs = pd.DataFrame(
    np.sort(long_df_[['phenotype_1', 'phenotype_2']].values, axis=1),
    columns=['ph_min', 'ph_max']
)
sorted_pairs=sorted_pairs.value_counts().reset_index()

data = sorted_pairs.pivot(index='ph_min', columns='ph_max', values='count')

sns.clustermap(data.fillna(data.T), 
               z_score=None, figsize=(6,6), xticklabels=True, yticklabels=True, col_cluster=True, row_cluster=True, 
               dendrogram_ratio=0.2, linewidths=0.5, cmap='Reds', linecolor='white')
plt.savefig(f"{saveloc}{cn_id}-hm_N.pdf", dpi=300, bbox_inches='tight') # Saves with high resolution and minimal whitespace
plt.show() 

In [ ]:
# Log overlap

long_df_ = long_df[(~long_df.phenotype_1.isna())&(~long_df.phenotype_2.isna())]
sorted_pairs = pd.DataFrame(
    np.sort(long_df_[['phenotype_1', 'phenotype_2']].values, axis=1),
    columns=['ph_min', 'ph_max']
)
sorted_pairs=sorted_pairs.value_counts().reset_index()

data = sorted_pairs.pivot(index='ph_min', columns='ph_max', values='count')

sns.clustermap(np.log10(data.fillna(data.T)), 
               z_score=None, figsize=(6,6), xticklabels=True, yticklabels=True, col_cluster=True, row_cluster=True, 
               dendrogram_ratio=0.2, linewidths=0.5, cmap='coolwarm', linecolor='white')
plt.savefig(f"{saveloc}{cn_id}-hm_log10N.pdf", dpi=300, bbox_inches='tight') # Saves with high resolution and minimal whitespace
plt.show()

#### EMD

In [ ]:
#1414 == 2000
#1414/200 == 2000/200 = 100
1414/200

In [ ]:
out_dict = {}

cn_id = "EMD_5clus"

cn_cells_data = pd.read_csv(f'/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/{cn_id}-cells_data.csv')
obs_data = sc.read('/mnt/disks/data/imc/CART_cohort/out/emd_work/emd_clustering-all.h5ad').obs

from tqdm import tqdm

for sample_id in tqdm(cn_cells_data['sample_id'].unique()):

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]    
    #obs_sample['ObjectNumber'] = obs_sample.index.astype(int)

    #merge cn
    cn_cells_sample = cn_cells_data[cn_cells_data.sample_id==sample_id]
    obs_sample = obs_sample.merge(cn_cells_sample[['ObjectNumber','cn_labels']],how='left')

    obs_sample['Cell_ID'] = obs_sample['ObjectNumber']
    obs_sample['x'] = obs_sample['Pos_X']
    obs_sample['y'] = obs_sample['Pos_Y']

    # Standard version (very fast for <10k cells)
    border_matrix, cell_ids = calculate_average_border_distances_centroids(obs_sample, max_distance=7)

    long_df = coo_to_long_df_vectorized(
        border_matrix, cell_ids, obs_sample, 
        phenotype_col='cn_labels',
        additional_cols=None
    )
    
    long_df['sample_id'] = sample_id
    
    out_dict[sample_id] = long_df
    
long_df = pd.concat(out_dict.values(),axis=0)
long_df.to_csv(f"{saveloc}{cn_id}-dist_data.csv",index=False)

In [ ]:
# Log overlap

saveloc = '/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/'

long_df_ = long_df[(~long_df.phenotype_1.isna())&(~long_df.phenotype_2.isna())]
sorted_pairs = pd.DataFrame(
    np.sort(long_df_[['phenotype_1', 'phenotype_2']].values, axis=1),
    columns=['ph_min', 'ph_max']
)
sorted_pairs=sorted_pairs.value_counts().reset_index()

data = sorted_pairs.pivot(index='ph_min', columns='ph_max', values='count')

sns.clustermap(np.log10(data.fillna(data.T)), 
               z_score=None, figsize=(4,4), xticklabels=True, yticklabels=True, col_cluster=True, row_cluster=True, 
               dendrogram_ratio=0.2, linewidths=0.5, cmap='coolwarm', linecolor='white')
plt.savefig(f"{saveloc}{cn_id}-hm_log10N.svg", dpi=300, bbox_inches='tight') # Saves with high resolution and minimal whitespace
plt.show()

### EMD: distances

* CN0 (pure PC) contacts both CN4 (prolif PC) and the mixed immune/PC CN.
* Whereas CN4 is isolated from mixed immune/PC CN.

In [ ]:
#1414px=2000um
#1414/10px = 141px = 2000/10 = 200um


In [ ]:
out_dict = {}

cn_id = "EMD_5clus"

cn_cells_data = pd.read_csv(f'/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/{cn_id}-cells_data.csv')
obs_data = sc.read('/mnt/disks/data/imc/CART_cohort/out/emd_work/emd_clustering-all.h5ad').obs

d_px=70

from tqdm import tqdm

for sample_id in tqdm(cn_cells_data['sample_id'].unique()):

    #import
    obs_sample = obs_data[obs_data.sample_id==sample_id]    
    #obs_sample['ObjectNumber'] = obs_sample.index.astype(int)

    #merge cn
    cn_cells_sample = cn_cells_data[cn_cells_data.sample_id==sample_id]
    obs_sample = obs_sample.merge(cn_cells_sample[['ObjectNumber','cn_labels']],how='left')

    obs_sample['Cell_ID'] = obs_sample['ObjectNumber']
    obs_sample['x'] = obs_sample['Pos_X']
    obs_sample['y'] = obs_sample['Pos_Y']

    # Standard version (very fast for <10k cells)
    border_matrix, cell_ids = calculate_average_border_distances_centroids(obs_sample, max_distance=d_px)

    long_df = coo_to_long_df_vectorized(
        border_matrix, cell_ids, obs_sample, 
        phenotype_col='cn_labels',
        additional_cols=None
    )
    
    long_df['sample_id'] = sample_id
    
    out_dict[sample_id] = long_df
    
long_df = pd.concat(out_dict.values(),axis=0)

# Calculate pairwise, normalize

long_df_ = long_df[(~long_df.phenotype_1.isna())&(~long_df.phenotype_2.isna())]

sorted_pairs = pd.DataFrame(
    np.sort(long_df_[['phenotype_1', 'phenotype_2']].values, axis=1),
    columns=['ph_min', 'ph_max']
)
sorted_pairs['sample_id'] = long_df_['sample_id'].tolist()

sorted_pairs = sorted_pairs.value_counts().reset_index()
sorted_pairs['ph_min_max'] = sorted_pairs['ph_min'].astype(str)+'_'+sorted_pairs['ph_max'].astype(str)

sample_phenotype_totals = pd.concat([
    sorted_pairs.groupby(['sample_id','ph_min'])['count'].sum(),
    sorted_pairs.groupby(['sample_id','ph_max'])['count'].sum()
]).groupby(level=[0,1]).sum()

sorted_pairs['norm_counts'] = sorted_pairs.apply(
    lambda row: row['count'] / np.sqrt(
        sample_phenotype_totals[row['sample_id']][row['ph_min']] * sample_phenotype_totals[row['sample_id']][row['ph_max']]
    ), axis=1
)
saveloc = '/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/'
sorted_pairs.to_csv(f"{saveloc}{cn_id}-CN_dist-{d_px}.csv",index=False)

In [ ]:
saveloc = '/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/'
cn_id = "EMD_5clus"
d_px=70
f"{saveloc}{cn_id}-CN_dist-{d_px}.csv"